# Class Shape Transformation

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.special import comb
from scipy.optimize import least_squares
from pathlib import Path
import warnings
import time

INPUT_FOLDER = Path.home() / "Downloads" / "UIUC_Airfoils"
OUTPUT_FOLDER = Path.home() / "Downloads" / "UIUC_CST_Output_12"
N_PARAMS = 8

OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)
(OUTPUT_FOLDER / "plots").mkdir(exist_ok=True)
warnings.filterwarnings("ignore")


 N_Params indicate the number of parameters. More parameters allow for more complex shapes but too many of them constraints the airfoil designer. Filterwarnings is used to remove basic errors like division by 0 to avoid any runtime issues throughout the parameterization of the airfoil.

In [ ]:
class PhysicCSTFitter:

    def __init__(self, n_params=8):
        self.n = n_params - 1
        self.n_params = n_params

    def bernstein_basis(self, x):
        B = np.zeros((len(x), self.n_params))
        for k in range(self.n_params):
            B[:, k] = comb(self.n, k) * (x**k) * ((1 - x)**(self.n - k))
        return B

    def cst_curve(self, x, weights, dz_te, is_upper=True):
        C = np.sqrt(x) * (1 - x)
        S = self.bernstein_basis(x) @ weights

        if is_upper:
            y = C * S + x * (dz_te / 2.0)
        else:
            y = -C * S - x * (dz_te / 2.0)
        return y

    def objective_function(self, params, x_u, y_u, x_l, y_l):

        w0 = params[0]
        w_u = np.concatenate(([w0], params[1:self.n_params]))
        w_l = np.concatenate(([w0], params[self.n_params : 2*self.n_params - 1]))
        dz = params[-1]

        y_u_fit = self.cst_curve(x_u, w_u, dz, is_upper=True)
        y_l_fit = self.cst_curve(x_l, w_l, dz, is_upper=False)

        res_u = y_u_fit - y_u
        res_l = y_l_fit - y_l

        weight_u = 1.0 + 5.0 * np.exp(-20 * x_u)
        weight_l = 1.0 + 5.0 * np.exp(-20 * x_l)

        residuals = np.concatenate((res_u * weight_u, res_l * weight_l))

        reg_u = np.diff(np.diff(w_u)) * 0.05
        reg_l = np.diff(np.diff(w_l)) * 0.05

        slope_u_te = -w_u[-1] + dz/2.0
        slope_l_te = w_l[-1] - dz/2.0

        kutta_penalty = []
        if slope_u_te > 0.0: kutta_penalty.append((slope_u_te) * 10.0)
        if slope_l_te < 0.0: kutta_penalty.append((abs(slope_l_te)) * 10.0)

        return np.concatenate((residuals, reg_u, reg_l, kutta_penalty))

    def fit_airfoil(self, upper_pts, lower_pts):
        x_u, y_u = upper_pts[:, 0], upper_pts[:, 1]
        x_l, y_l = lower_pts[:, 0], lower_pts[:, 1]

        dz_est = max(0.0, abs(y_u[-1] - y_l[-1]))

        initial_guess = np.ones(1 + 2*(self.n_params-1) + 1) * 0.15
        initial_guess[-1] = dz_est

        lb = [0.1] + [-0.5]*14 + [0.0]
        ub = [0.6] + [ 0.8]*14 + [0.02]

        try:
            res = least_squares(
                self.objective_function,
                initial_guess,
                bounds=(lb, ub),
                args=(x_u, y_u, x_l, y_l),
                method='trf',
                loss='soft_l1'
            )
        except ValueError:
            return {"status": "FAILED_OPTIMIZATION"}

        p = res.x
        w0 = p[0]
        w_u = np.concatenate(([w0], p[1:self.n_params]))
        w_l = np.concatenate(([w0], p[self.n_params : 2*self.n_params - 1]))
        dz = p[-1]

        if res.cost > 0.02:
            status = "POOR_FIT"
        else:
            status = "SUCCESS"

        return {
            "w_u": w_u, "w_l": w_l, "dz": dz,
            "status": status, "cost": res.cost,
            "fit_u": (x_u, self.cst_curve(x_u, w_u, dz, True)),
            "fit_l": (x_l, self.cst_curve(x_l, w_l, dz, False))
        }

**self.n** = degree of the polynomial. It is denoted by no. of parameters $- 1 = 7$ in this case.

CST builds an airfoil shape by adding up several "basis" curves. These curves are Bernstein polynomials (similar to those used in Bezier curves in graphic design).

$$B_{k,n}(x) = \binom{n}{k} x^k (1-x)^{n-k}$$

**bernstein_basis()** returns a matrix where each column represents one basis curve evaluated at all $x$ points.

**cst_curve()** defines the CST Curve equation as:

*  $C$: The Class Function. $sqrt(x)$ creates a round nose (leading edge), and $(1-x)$ ensures the tail goes to zero. This defines the generic airfoil-like round-nose/sharp-tail shape.
* $S$: The Shape Function. This modifies the generic shape C to match the specific airfoil. It is the dot product of the Basis matrix B and the weights (coefficients).
* $dz_{te}$: Trailing Edge Thickness. Real airfoils don't end at a perfect point; they have a small thickness.
* Upper: $y = C \cdot S + \text{offset}$. Lower: Same but negative (going down).
* $x * (dz_{te} / 2.0)$ linearly adds thickness so the airfoil opens up slightly at the tail.

**objective_function()** tries to make the error zero.

* $w[0]$ is the first weight. It is shared between upper and lower surfaces. This ensures the Leading Edge radius is continuous.
* $w_u / w_l$ : Reconstructs the full list of weights for upper and lower surfaces.
* $dz$ : The trailing edge thickness parameter.
* Subtracts the actual data $y_u$ from the fitted data $y_{ufit}$ to find the error (residuals).
* $weight_u$ : Creates a multiplier array. If $x$ is near $0$ (nose), the weight is high $1 + 5 = 6x$ importance. If $x$ is near $1$ (tail), the weight is low $1x$ importance. The front 10% of an airfoil (Leading Edge) controls stall and lift characteristics. It is more important than the flat middle part.
* **np.diff(np.diff(w_u))**: If the weights oscillate (e.g., 0.1, 0.9, 0.1), the surface will be wavy. Wavy surfaces crash CFD solvers. It calculates the second derivative of the weights list and penalizes high curvature to force the optimizer to choose "smooth" progressions of weights.
* **Kutta Condition** : In aerodynamics, airflow must leave the trailing edge smoothly. The upper surface slope must look "down" or flat, and the lower must look "up" or flat. They cannot "cross" or diverge outwards. $slope_{ute}$: Calculates the slope at $x=1$ analytically. if $slope_{ute} > 0.0$: If the upper surface is curling upwards at the tail, that's physically bad. Add a huge penalty $* 10.0$ so the optimizer stops doing that.

**fit_airfoil()** establishes the fitting routine of the weighed parameters.

* Sets all weights to 0.15 initially. This roughly looks like a standard tear-drop airfoil (NACA 0012). Starting with a decent shape prevents the math from exploding.
* Bounds put constraints on leading and trailing edge thickness.







In [ ]:
class BatchProcessor:
    def parse_file(self, path):
        try:
            with open(path, 'r', errors='ignore') as f:
                lines = [l.strip() for l in f.readlines() if l.strip()]

            coords = []
            for line in lines[1:]:
                parts = line.split()
                if len(parts) >= 2:
                    try: coords.append([float(parts[0]), float(parts[1])])
                    except: pass

            data = np.array(coords)
            if len(data) < 10: return None, None, None

            le_idx = np.argmin(data[:, 0])
            upper = data[:le_idx+1]
            lower = data[le_idx:]

            upper = np.flip(upper, axis=0)

            max_c = np.max(data[:, 0])
            upper /= max_c
            lower /= max_c

            return upper, lower, path.stem
        except:
            return None, None, None

    def run(self):
        fitter = PhysicCSTFitter(n_params=N_PARAMS)
        csv_data = []
        files = list(INPUT_FOLDER.glob("*.dat"))

        print(f"--- UIUC Database Processor (Enhanced Physics) ---")
        print(f"Input: {INPUT_FOLDER}")
        print(f"Output: {OUTPUT_FOLDER}")
        print(f"Found {len(files)} files. Processing...")

        start_time = time.time()

        success_count = 0
        for i, fpath in enumerate(files):
            u_pts, l_pts, name = self.parse_file(fpath)

            if u_pts is None: continue

            res = fitter.fit_airfoil(u_pts, l_pts)

            if res['status'] == "SUCCESS":
                success_count += 1
                entry = {
                    "Airfoil_Name": name,
                    "Fit_Error_SSE": round(res['cost'], 6),
                    "TE_Thickness_dz": round(res['dz'], 6)
                }

                for idx, w in enumerate(res['w_u']):
                    entry[f"wu_{idx}"] = round(w, 6)

                for idx, w in enumerate(res['w_l']):
                    entry[f"wl_{idx}"] = round(w, 6)

                csv_data.append(entry)

                if i < 50: # Plot first 50 to verify smoothing
                    self.plot_verify(name, u_pts, l_pts, res['fit_u'], res['fit_l'])

            if i % 50 == 0:
                print(f"Processed {i}/{len(files)} | Valid: {success_count}")

        df = pd.DataFrame(csv_data)

        cols = ['Airfoil_Name', 'Fit_Error_SSE', 'TE_Thickness_dz']
        cols += [f'wu_{j}' for j in range(N_PARAMS)]
        cols += [f'wl_{j}' for j in range(N_PARAMS)]
        df = df[cols]

        save_path = OUTPUT_FOLDER / "UIUC_CST_Database_12.csv"
        df.to_csv(save_path, index=False)

        print(f"\nDone! Processed {len(files)} files in {time.time()-start_time:.1f}s.")
        print(f"Successfully parameterized {len(df)} airfoils.")
        print(f"DATABASE SAVED TO: {save_path}")

    def plot_verify(self, name, u_raw, l_raw, u_fit, l_fit):
        plt.figure(figsize=(8, 3))
        plt.plot(u_raw[:,0], u_raw[:,1], 'k.', markersize=2, label='Raw')
        plt.plot(l_raw[:,0], l_raw[:,1], 'k.', markersize=2)
        plt.plot(u_fit[0], u_fit[1], 'r-', linewidth=1, label='CST (Smoothed)')
        plt.plot(l_fit[0], l_fit[1], 'r-', linewidth=1)
        plt.legend()
        plt.title(f"Fit Check: {name}")
        plt.axis("equal")
        plt.grid(True, alpha=0.3)
        plt.savefig(OUTPUT_FOLDER / "plots" / f"{name}_fit.png")
        plt.close()

**parse_file()** handles the file I/O.

* **Splitting**: Airfoil files usually run Back -> Front -> Back. $le_{idx}$: Finds the point with the smallest $x$ value (the Leading Edge / Nose). Splits the array into Upper (Top -> Nose) and Lower (Nose -> Bottom). flip: Ensures both arrays run from Nose $(0)$ to Tail $(1)$.
* **Normalization**: Divides by max_c (Chord Length). This ensures the airfoil ranges from $0.0$ to $1.0$, regardless of whether the original file was in meters, inches, or millimeters.

**run()** finds all the $.dat$ files in the folder. Loops through every file, parses it, and fits it using the **PhysicCSTFitter**.

In [ ]:
if __name__ == "__main__":
    if not INPUT_FOLDER.exists():
        print("Error: Airfoil folder not found. Run the downloader first.")
    else:
        BatchProcessor().run()

# Genetic Algorithm

**prefinalga.py** was the first file to be executed to create 7500 samples. It was found to reach its perfect airfoil sampling by the 50th generation, giving the highest fitness score and it stayed the same for the rest of the generations as the sampling evolved. This rendered a rather consistent set of samples which caused issues whilst training the model using PointNet++ CNN as it only learned from consistent "perfect" samples and its validation error values spiked as soon as it encountered any of the samples from the intial 50 generations.

In order to takcle this issue, an **NSGA-II core **implemented strategy was used to create 7500 samples. Unlike the previous Weighted Sum approach (which collapses everything into one number), NSGA-II maintains a Pareto Front. This means it finds a set of optimal solutions where you cannot improve one objective (e.g., Lift/Drag) without degrading another (e.g., Pitching Moment). This gives you a trade-off curve rather than a single "best" point, which is crucial for engineering decisions.

* True Multi-Objective: It optimizes Efficiency (L/D), Operating Range (Drag Bucket), and Stability (Moment Coeff) simultaneously and independently.
* Pareto Dominance: It uses rigorous non-dominated sorting logic (no external libraries required, I wrote the raw logic to ensure it runs without dependency errors).
* Diversity Preservation: It calculates "Crowding Distance" to ensure your solutions don't bunch up in one spot (e.g., it won't just give you 100 variations of the same high-lift airfoil; it will give you a range from high-lift to high-speed).
* Generational Loop: It switches from steady-state to a proper Generational P + Q truncation loop, which is more stable for complex physics problems.

This model wasn't converging as well on the PointNet++ CNN giving a 10x margin difference in the validation and training errors.

In order to counter the above, the issue was resolved by treating the Genetic Algorithm not just as an Optimizer but a diversifier as well. This was achieved by giving it a Reynolds number sweep to generate multiple samples of different Reynolds Numbers to make the CNN learn and validate effectively, giving errors of the same margin. This gave High Quality Data with High Variety.

**This is a significant architectural upgrade mpving from a Static Shape Analyzer to a Physics-Informed Surrogate Model.**

In [ ]:
import pandas as pd
import numpy as np
import h5py
import subprocess
import multiprocessing
import os
import sys
import time
import random
import shutil
import tempfile
from scipy.special import comb
import traceback

# --- CONFIGURATION ---
CONFIG = {
    "INPUT_CSV": "UIUC_CST_Database.csv",
    "OUTPUT_H5": "airfoil_dataset_re_sweep.h5",
    "N_CORES": 6,
    "TARGET_SAMPLES": 8000,
    "POPULATION_SIZE": 100,

    # REYNOLDS SWEEP RANGE
    "RE_MIN": 200000.0,
    "RE_MAX": 3000000.0,

    "MACH": 0.0,
    "ALPHA_SEQ": (-4.0, 16.0, 1.0),
    "ITERATIONS": 200,
    "XFOIL_TIMEOUT": 25.0,

    "THICKNESS_MIN": 0.05,
    "THICKNESS_MAX": 0.22,
    "DZ_TE_MAX": 0.015,
    "MAX_INFLECTIONS": 2,

    "CL_MAX_LIMIT": 2.2,
    "CD_MIN_LIMIT": 0.002,
}

def find_xfoil(): # To find and execute X-Foil
    if os.path.exists("xfoil.exe"): return os.path.abspath("xfoil.exe")
    path_exec = shutil.which("xfoil.exe")
    if path_exec: return path_exec
    common_paths = [r"C:\XFOIL\xfoil.exe", r"C:\Program Files\XFOIL\xfoil.exe", "/usr/bin/xfoil", "/usr/local/bin/xfoil"]
    for p in common_paths:
        if os.path.exists(p): return p
    return None

XFOIL_PATH = find_xfoil()

class RobustCST: # Performs CST parameterization on the generation .dat files from the samples of genetic algorithm.
    def __init__(self, n_params=8, resolution=200):
        self.n_params = n_params
        self.resolution = resolution
        self.beta = np.linspace(0, np.pi, self.resolution)
        self.x = 0.5 * (1 - np.cos(self.beta))
        self.B = np.zeros((self.resolution, self.n_params))
        self.C = np.sqrt(self.x) * (1 - self.x)
        n = self.n_params - 1
        for k in range(self.n_params):
            self.B[:, k] = comb(n, k) * (self.x**k) * ((1 - self.x)**(n - k))

    def check_curvature(self, y_coords):
        curv = np.gradient(np.gradient(y_coords))
        sign_changes = np.diff(np.sign(curv))
        return np.count_nonzero(sign_changes) <= CONFIG["MAX_INFLECTIONS"]

    def generate(self, w_u, w_l, dz_te):
        w_l = np.array(w_l, dtype=np.float64)
        w_u = np.array(w_u, dtype=np.float64)
        w_l[0] = w_u[0]

        S_u = self.B @ w_u
        S_l = self.B @ w_l
        y_u = self.C * S_u + self.x * (dz_te / 2.0)
        y_l = -self.C * S_l - self.x * (dz_te / 2.0)

        slope_u_TE = -w_u[-1] + (dz_te / 2.0)
        slope_l_TE = w_l[-1] - (dz_te / 2.0)
        if slope_u_TE > 0.02 or slope_l_TE < -0.02: return None, None, "INVALID: Diverging TE"

        thickness = y_u - y_l
        if np.any(thickness < -1e-6): return None, None, "INVALID: Cross"

        max_t = np.max(thickness)
        if max_t < CONFIG["THICKNESS_MIN"]: return None, None, "INVALID: Too Thin"
        if max_t > CONFIG["THICKNESS_MAX"]: return None, None, "INVALID: Too Thick"

        if not self.check_curvature(y_u) or not self.check_curvature(y_l): return None, None, "INVALID: Wavy"

        x_coords = np.concatenate((self.x[::-1], self.x[1:]))
        y_coords = np.concatenate((y_u[::-1], y_l[1:]))
        return x_coords, y_coords, "VALID"

def worker_task(task_data):
    if XFOIL_PATH is None: return None
    job_id, w_u, w_l, dz = task_data

    geo = RobustCST()
    x, y, status = geo.generate(w_u, w_l, dz)
    if status != "VALID": return None

    # --- REYNOLDS SWEEP LOGIC ---
    # Pick a random Re for this specific evaluation
    # This forces the GA to keep shapes that work across various speeds
    current_re = random.uniform(CONFIG['RE_MIN'], CONFIG['RE_MAX'])

    result = None
    with tempfile.TemporaryDirectory() as tmp_dir:
        f_dat = os.path.join(tmp_dir, "airfoil.dat")
        f_log = os.path.join(tmp_dir, "polar.log")
        f_cp  = os.path.join(tmp_dir, "cp_dist.txt")

        try:
            with open(f_dat, 'w') as f:
                f.write(f"AF_{job_id}\n")
                for i in range(len(x)):
                    f.write(f" {x[i]:.6f}  {y[i]:.6f}\n")

            cmds_sweep = (
                f"load {f_dat}\n"
                "ppar\n" "N 200\n" "pane\n" "\n" "\n"
                "oper\n"
                f"v {current_re}\n" # Use the randomized Re
                f"mach {CONFIG['MACH']}\n"
                f"iter {CONFIG['ITERATIONS']}\n"
                "pacc\n" f"{f_log}\n" "\n"
                f"aseq {CONFIG['ALPHA_SEQ'][0]} {CONFIG['ALPHA_SEQ'][1]} {CONFIG['ALPHA_SEQ'][2]}\n"
                "pacc\n" "quit\n"
            )
            subprocess.run(XFOIL_PATH, input=cmds_sweep, text=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, timeout=CONFIG['XFOIL_TIMEOUT'], cwd=tmp_dir)

            polar_data = []
            best_stats = {'alpha': 0, 'cl': 0, 'cd': 1.0, 'ld': -1.0, 'cm': 0.0}

            if os.path.exists(f_log):
                with open(f_log, 'r') as f:
                    for line in f:
                        vals = line.split()
                        if len(vals) >= 7 and not any(c.isalpha() for c in vals[0]) and "#" not in line:
                            try:
                                a, cl, cd, cm = float(vals[0]), float(vals[1]), float(vals[2]), float(vals[4])
                                if (cd > CONFIG['CD_MIN_LIMIT'] and abs(cl) < CONFIG['CL_MAX_LIMIT'] and cd < 1.0):
                                    polar_data.append({'alpha': a, 'cl': cl, 'cd': cd, 'cm': cm})
                                    ld = cl / cd
                                    if ld > best_stats['ld']:
                                        best_stats = {'alpha': a, 'cl': cl, 'cd': cd, 'ld': ld, 'cm': cm}
                            except ValueError: continue

            if len(polar_data) > 3 and best_stats['ld'] > 5.0:

                # Objectives: Efficiency, Stall Angle, Pitching Moment
                obj_ld = best_stats['ld']
                obj_stall = best_stats['alpha']
                obj_cm = -abs(best_stats['cm'])

                cmds_cp = (
                    f"load {f_dat}\n" "ppar\n" "N 200\n" "pane\n" "\n" "\n"
                    "oper\n" f"v {current_re}\n" f"mach {CONFIG['MACH']}\n" f"iter {CONFIG['ITERATIONS']}\n"
                    f"alfa {best_stats['alpha']}\n" f"cpwr {f_cp}\n" "quit\n"
                )
                subprocess.run(XFOIL_PATH, input=cmds_cp, text=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, timeout=5.0, cwd=tmp_dir)

                cp_fixed = np.zeros(200)
                if os.path.exists(f_cp):
                    try:
                        raw_cp = np.loadtxt(f_cp, skiprows=3)
                        if raw_cp.ndim == 2 and raw_cp.shape[0] > 10:
                            cp_fixed = np.interp(np.linspace(0, 1, 200), raw_cp[:,0], raw_cp[:,2])
                    except: pass

                result = {
                    'w_u': w_u, 'w_l': w_l, 'dz': dz,
                    'cl': best_stats['cl'], 'cd': best_stats['cd'], 'cm': best_stats['cm'],
                    'alpha_opt': best_stats['alpha'],
                    'reynolds': current_re, # Save the Re used!
                    'cp': cp_fixed,
                    'objectives': [obj_ld, obj_stall, obj_cm]
                }
        except Exception: pass
    return result

# --- NSGA-II CORE (Preserved from previous optimal code) ---
class NSGA2_Core:
    @staticmethod
    def dominates(p_objs, q_objs):
        better_or_equal = all(x >= y for x, y in zip(p_objs, q_objs))
        strictly_better = any(x > y for x, y in zip(p_objs, q_objs))
        return better_or_equal and strictly_better

    @staticmethod
    def fast_non_dominated_sort(population):
        fronts = [[]]
        for p in population:
            p['domination_count'] = 0
            p['dominated_solutions'] = []
            for q in population:
                if NSGA2_Core.dominates(p['objectives'], q['objectives']):
                    p['dominated_solutions'].append(q)
                elif NSGA2_Core.dominates(q['objectives'], p['objectives']):
                    p['domination_count'] += 1
            if p['domination_count'] == 0:
                p['rank'] = 0
                fronts[0].append(p)
        i = 0
        while len(fronts[i]) > 0:
            next_front = []
            for p in fronts[i]:
                for q in p['dominated_solutions']:
                    q['domination_count'] -= 1
                    if q['domination_count'] == 0:
                        q['rank'] = i + 1
                        next_front.append(q)
            i += 1
            if next_front: fronts.append(next_front)
            else: break
        return fronts

    @staticmethod
    def crowding_distance_assignment(front):
        if len(front) == 0: return
        l = len(front)
        num_obj = len(front[0]['objectives'])
        for p in front: p['distance'] = 0.0
        for m in range(num_obj):
            front.sort(key=lambda x: x['objectives'][m])
            front[0]['distance'] = float('inf')
            front[-1]['distance'] = float('inf')
            scale = front[-1]['objectives'][m] - front[0]['objectives'][m]
            if scale == 0: continue
            for i in range(1, l - 1):
                front[i]['distance'] += (front[i+1]['objectives'][m] - front[i-1]['objectives'][m]) / scale

class GeneticEngine:
    def __init__(self):
        self.population = []

    def load_initial_seeds(self):
        if not os.path.exists(CONFIG['INPUT_CSV']):
            print("CRITICAL: Input CSV not found."); sys.exit(1)
        try:
            df = pd.read_csv(CONFIG['INPUT_CSV'])
            df.columns = df.columns.str.strip()
            w_u_cols = [f'wu_{i}' for i in range(8)]
            w_l_cols = [f'wl_{i}' for i in range(8)]
            for _, row in df.iterrows():
                self.population.append({
                    'w_u': row[w_u_cols].values.astype(float),
                    'w_l': row[w_l_cols].values.astype(float),
                    'dz': float(row['TE_Thickness_dz']),
                    'objectives': [0,0,0], 'rank':0, 'distance':0
                })
            print(f"Loaded {len(self.population)} seeds.")
        except Exception: traceback.print_exc(); sys.exit(1)

    def save_batch(self, h5f, batch, datasets):
        if not batch: return
        n = len(batch)
        w = np.array([np.concatenate([r['w_u'], r['w_l'], [r['dz']]]) for r in batch])
        # Note: Added 'reynolds' to scalars array (5 columns now)
        s = np.array([[r['cl'], r['cd'], r['cm'], r['alpha_opt'], r['reynolds']] for r in batch])
        c = np.array([r['cp'] for r in batch])

        idx = datasets['w'].shape[0]
        datasets['w'].resize(idx+n, axis=0); datasets['w'][idx:] = w
        datasets['s'].resize(idx+n, axis=0); datasets['s'][idx:] = s
        datasets['c'].resize(idx+n, axis=0); datasets['c'][idx:] = c
        h5f.flush()

    def crowded_tournament_selection(self):
        p1 = random.choice(self.population)
        p2 = random.choice(self.population)
        if p1['rank'] < p2['rank']: return p1
        elif p2['rank'] < p1['rank']: return p2
        else:
            if p1['distance'] > p2['distance']: return p1
            else: return p2

    def run(self):
        self.load_initial_seeds()

        with h5py.File(CONFIG['OUTPUT_H5'], 'w') as h5f:
            datasets = {
                'w': h5f.create_dataset("weights", (0, 17), maxshape=(None, 17)),
                # Increased to 5 columns to store Reynolds Number
                's': h5f.create_dataset("scalars", (0, 5), maxshape=(None, 5)),
                'c': h5f.create_dataset("cp", (0, 200), maxshape=(None, 200))
            }
            pool = multiprocessing.Pool(CONFIG['N_CORES'])

            print("--- Initializing Population ---")
            tasks = [(i, p['w_u'], p['w_l'], p['dz']) for i, p in enumerate(self.population)]
            # Run init tasks
            results = pool.map(worker_task, tasks[:CONFIG['POPULATION_SIZE']*2])

            valid_pop = [r for r in results if r is not None]

            # Initial NSGA-II Sort
            fronts = NSGA2_Core.fast_non_dominated_sort(valid_pop)
            for f in fronts: NSGA2_Core.crowding_distance_assignment(f)

            # Truncate
            self.population = []
            for f in fronts:
                if len(self.population) + len(f) <= CONFIG['POPULATION_SIZE']:
                    self.population.extend(f)
                else:
                    f.sort(key=lambda x: x['distance'], reverse=True)
                    self.population.extend(f[:CONFIG['POPULATION_SIZE'] - len(self.population)])
                    break

            self.save_batch(h5f, valid_pop, datasets)
            total_saved = len(valid_pop)
            gen = 0

            print(f"\nSTARTING RE-SWEEP EVOLUTION | Target: {CONFIG['TARGET_SAMPLES']}")

            while total_saved < CONFIG['TARGET_SAMPLES']:
                gen += 1
                offspring_tasks = []

                while len(offspring_tasks) < CONFIG['POPULATION_SIZE']:
                    p1 = self.crowded_tournament_selection()
                    p2 = self.crowded_tournament_selection()

                    alpha = random.random()
                    c_u = p1['w_u']*alpha + p2['w_u']*(1-alpha)
                    c_l = p1['w_l']*alpha + p2['w_l']*(1-alpha)
                    c_dz = (p1['dz'] + p2['dz']) / 2.0

                    if random.random() < 0.2:
                        noise = np.random.normal(0, 0.05, 8)
                        c_u += noise
                        c_l += noise
                        c_dz += np.random.normal(0, 0.002)
                        c_l[0] = c_u[0]

                    c_dz = max(0.0, min(c_dz, CONFIG['DZ_TE_MAX']))
                    offspring_tasks.append((random.randint(0, 1e9), c_u, c_l, c_dz))

                batch_results = pool.map(worker_task, offspring_tasks)
                valid_offspring = [r for r in batch_results if r is not None]

                if valid_offspring:
                    self.save_batch(h5f, valid_offspring, datasets)
                    total_saved += len(valid_offspring)

                    combined_pop = self.population + valid_offspring
                    fronts = NSGA2_Core.fast_non_dominated_sort(combined_pop)
                    for f in fronts: NSGA2_Core.crowding_distance_assignment(f)

                    next_gen = []
                    for f in fronts:
                        if len(next_gen) + len(f) <= CONFIG['POPULATION_SIZE']:
                            next_gen.extend(f)
                        else:
                            f.sort(key=lambda x: x['distance'], reverse=True)
                            next_gen.extend(f[:CONFIG['POPULATION_SIZE'] - len(next_gen)])
                            break
                    self.population = next_gen

                    print(f"Gen {gen:03d} | Saved: +{len(valid_offspring):02d} | Total: {total_saved:04d} | "
                          f"Re Range: {min(r['reynolds'] for r in valid_offspring):.0f}-{max(r['reynolds'] for r in valid_offspring):.0f}")

            pool.close(); pool.join()
            print(f"\n--- SUCCESS. Dataset saved to {CONFIG['OUTPUT_H5']} ---")

if __name__ == "__main__":
    multiprocessing.freeze_support()
    try:
        GeneticEngine().run()
    except KeyboardInterrupt:
        print("\nInterrupt.")
    except Exception:
        traceback.print_exc()

**worker_task()** is a great value addition function in this code.

* Instead of testing every airfoil at the same speed, it picks a random Reynolds number for this specific evaluation. This forces the Genetic Algorithm to learn shapes that are robust across all speeds, rather than optimizing for just one.
* Creates a temporary directory to store XFOIL input/output files safely without file name collisions between parallel workers.
* Constructs a string of commands (cmds_sweep) to send to XFOIL: "Load file", "Normalize", "Set Re", "Run Angle of Attack Sequence (aseq)".
* **subprocess.run**: actually launches XFOIL, pipes the commands in, and waits for it to finish.
* Reads polar.log.Extracts $C_l$ (Lift), $C_d$ (Drag), $C_m$ (Moment).Calculates $L/D$ (Lift-to-Drag ratio, i.e., Efficiency). Tracks the "best" statistics (highest efficiency) found during the sweep.
* If the airfoil was good ($L/D > 5$), runs XFOIL again at the specific optimal angle of attack (alpha_opt) to get the Pressure Coefficient ($C_p$) distribution. Interpolates the $C_p$ data to a fixed size of 200 points for consistent storage.
* Returns a dictionary containing the design parameters (weights), performance scalars ($C_l, C_d, Re$), $C_p$ distribution, and the 3 Objectives for optimization: Maximize $L/D$, Maximize Stall Angle, Minimize Pitching Moment ($C_m$).

**NSGA-2** implements the logic for the "Non-dominated Sorting Genetic Algorithm II".

* Compares two solutions (P and Q). P "dominates" Q if P is no worse than Q in all objectives AND strictly better in at least one.
* Sorts the population into "Fronts". Front 0: The best solutions (Pareto optimal) that are not dominated by anyone. Front 1: Solutions dominated only by Front 0, etc.
* Calculates how close a solution is to its neighbors in objective space.
Used to preserve diversity. If two solutions are equally good (same rank), NSGA-II prefers the one in the less crowded region.


# PointNet++ Framework

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import h5py
import os
import sys
import shutil
import tempfile
import random
import subprocess
import matplotlib.pyplot as plt
from scipy.special import comb
from sklearn.preprocessing import StandardScaler, MinMaxScaler

# --- CONFIGURATION ---
CONFIG = {
    "MODEL_PATH": "best_physics_pointnet.pth",  # File path to the trained AI model weights
    "DATASET_PATH": "airfoil_dataset_re_sweep.h5", # File path to data used to calibrate scalers

    # Target
    "TARGET_REYNOLDS": 1000000.0,  # The flight condition (speed/viscosity) we are designing for
    "TARGET_CL": 0.85,             # The Lift Coefficient we want to achieve
    "MIN_THICKNESS": 0.11,         # Constraint: Airfoil must be at least 11% thick (for fuel/spars)

    # Robustness Constraints (THE FIX)
    "MIN_LE_RADIUS": 0.015,        # Constraint: Leading edge must be round enough to prevent stalling
    "MAX_CURVATURE_ENERGY": 5.0,   # Constraint: Surface must be smooth (no waves/bumps)

    "POPULATION_SIZE": 1000,       # Genetic Algo: How many airfoils to test in each generation
    "GENERATIONS": 150,            # Genetic Algo: How many evolution loops to run
    "DEVICE": "cuda" if torch.cuda.is_available() else "cpu" # Use NVIDIA GPU if available, else CPU
}

def index_points(points, idx):
    B, C, N = points.shape
    S, K = idx.shape[1], idx.shape[2] # idx is a tensor containing the indices of the points to be extracted.
    points_t = points.transpose(2, 1).contiguous() # Swaps dimensions so "Channels" (x,y,z features) are at the end.
    flat_points = points_t.view(B * N, C)
    offset = torch.arange(B, device=points.device).view(B, 1, 1) * N # Creates an index offset (0, N, 2N...) so that indices for Batch 1 don't accidentally grab points from Batch 0.
    flat_idx = (idx + offset).view(-1)
    res = flat_points[flat_idx]
    return res.view(B, S, K, C)

**index_points()** is used to gather points from the Point Cloud. B, C, N represent Batch Size, Channels (x,y,z) and Number of points. S is the number of groups, K is the number of neighbors. It flattens the batch of points into one long list, calculates the correct indices (adjusting for batch offsets), retrieves the data, and reshapes it back. This allows the network to say "Give me the 32 neighbors for Point #5".

In [ ]:
class PointNetSetAbstraction(nn.Module):
    def __init__(self, npoint, radius, nsample, in_channel, mlp, group_all):
        super(PointNetSetAbstraction, self).__init__()
        self.npoint = npoint # Number of points to target
        self.radius = radius # How far to look for neighbors
        self.nsample = nsample # Number of neighbors to group together
        self.group_all = group_all # if True, groups all points into one single vector
        self.mlp_convs = nn.ModuleList() # A list of Convolutional Layers.
        self.mlp_bns = nn.ModuleList() # A list of Normalization Layers
        last_channel = in_channel + 3 # Adding 3 because the code appends 3 coordinates (x, y, z) to the features
        for out_channel in mlp:
            self.mlp_convs.append(nn.Conv2d(last_channel, out_channel, 1))
            self.mlp_bns.append(nn.BatchNorm2d(out_channel))
            last_channel = out_channel

    def forward(self, xyz, points):
        B, C, N = xyz.shape
        if self.group_all:
            if points is not None: new_points = torch.cat([xyz, points], dim=1)
            else: new_points = xyz
            new_xyz = None
            new_points = new_points.unsqueeze(2)
        else:
            stride = N // self.npoint
            idx = torch.arange(0, N, stride, device=xyz.device)[:self.npoint]
            new_xyz = xyz[:, :, idx]
            dist = torch.cdist(new_xyz.transpose(1,2), xyz.transpose(1,2))
            val, group_idx = torch.topk(dist, self.nsample, dim=2, largest=False)
            grouped_xyz = index_points(xyz, group_idx).permute(0, 3, 1, 2)
            grouped_xyz -= new_xyz.transpose(1,2).unsqueeze(2).permute(0, 3, 1, 2)
            if points is not None:
                grouped_points = index_points(points, group_idx).permute(0, 3, 1, 2)
                new_points = torch.cat([grouped_xyz, grouped_points], dim=1)
            else: new_points = grouped_xyz

        for i, conv in enumerate(self.mlp_convs):
            bn = self.mlp_bns[i]
            new_points = F.relu(bn(conv(new_points)))
        new_points = torch.max(new_points, 3)[0]
        return new_xyz, new_points


This layer performs Downsampling (reducing point count) and Feature Encoding (learning shapes).

forward() is used to specify the data flow.

1. Sampling: Sampling (Farthest Point Sampling): Uses stride slicing to pick npoint centroids that are spread out across the shape.
2. Grouping (torch.cdist + topk): Calculates distances between all points and the centroids. Selects the nsample closest points for each centroid.
3. Relative Coordinates: grouped_xyz -= new_xyz.... Converts absolute coordinates to relative ones (e.g., "neighbor is 0.1 units to the right of centroid"). This makes the network understand local shape regardless of where the airfoil is in space.
4. Feature Extraction: Passes the grouped points through the MLP (self.mlp_convs).
5. Max Pooling: torch.max(new_points, 3)[0]. Takes the strongest feature signal from the local neighborhood. This aggregates the group into a single representation.

Returns:

new_xyz: The coordinates of the sampled centroids.

new_points: The learned features for those centroids.


In [ ]:
class PointNetFeaturePropagation(nn.Module):
    def __init__(self, in_channel, mlp):
        super(PointNetFeaturePropagation, self).__init__()
        self.mlp_convs = nn.ModuleList()
        self.mlp_bns = nn.ModuleList()
        last_channel = in_channel
        for out_channel in mlp:
            self.mlp_convs.append(nn.Conv1d(last_channel, out_channel, 1))
            self.mlp_bns.append(nn.BatchNorm1d(out_channel))
            last_channel = out_channel

    def forward(self, xyz1, xyz2, points1, points2):
        if xyz2 is None: interpolated_points = points2.repeat(1, 1, xyz1.shape[2])
        else:
            dist = torch.cdist(xyz1.transpose(1,2), xyz2.transpose(1,2))
            dists, idx = torch.topk(dist, 3, dim=2, largest=False)
            weight = 1.0 / (dists + 1e-10)
            weight = weight / torch.sum(weight, dim=2, keepdim=True)
            grouped = index_points(points2, idx)
            interpolated_points = torch.sum(grouped * weight.unsqueeze(-1), dim=2)
            interpolated_points = interpolated_points.transpose(1, 2)

        if points1 is not None: new_points = torch.cat([points1, interpolated_points], dim=1)
        else: new_points = interpolated_points

        for i, conv in enumerate(self.mlp_convs):
            bn = self.mlp_bns[i]
            new_points = F.relu(bn(conv(new_points)))
        return new_points

This layer performs Upsampling. It helps the network map global decisions back onto specific points on the airfoil surface.

forward () interpolates data from a sparse layer (xyz2) to a dense layer (xyz1).
1. Distance Calculation: Computes distance between dense points (xyz1) and sparse points (xyz2).
2. Weighting: Finds the 3 closest sparse points for every dense point. Calculates weights using Inverse Distance ($1/distance$), so closer points have more influence.
3. Interpolation: torch.sum(grouped * weight...). Computes the weighted average of features.
4. Concatenation: Joins the interpolated features with the existing features (points1) of the dense layer.
5. Refinement: Passes the result through the MLP to smooth out the features.

Returns: new_points (Dense features mapped back to the original point resolution).

In [ ]:
class ReAwareAirfoilPointNet(nn.Module):
    def __init__(self):
        super(ReAwareAirfoilPointNet, self).__init__()
        # Matches 'High-Precision' reduced architecture [64,64,128]
        self.sa1 = PointNetSetAbstraction(npoint=100, radius=0.1, nsample=32, in_channel=0, mlp=[64, 64, 128], group_all=False)
        self.sa2 = PointNetSetAbstraction(npoint=32, radius=0.2, nsample=32, in_channel=128, mlp=[128, 128, 256], group_all=False)
        self.sa3 = PointNetSetAbstraction(npoint=None, radius=None, nsample=None, in_channel=256, mlp=[256, 512, 1024], group_all=True)

        self.fp3 = PointNetFeaturePropagation(in_channel=1025+256, mlp=[256, 256])
        self.fp2 = PointNetFeaturePropagation(in_channel=256+128, mlp=[256, 128])
        self.fp1 = PointNetFeaturePropagation(in_channel=128+3, mlp=[128, 128, 128])

        self.conv_cp = nn.Conv1d(128, 1, 1) # Predict the pressure coefficient curve.

        self.fc1 = nn.Linear(1025, 512) # Fully Connected Dense layers to predict CL, CD, CM
        self.bn1 = nn.BatchNorm1d(512)
        self.drop1 = nn.Dropout(0.4)
        self.fc2 = nn.Linear(512, 256) # Fully Connected Dense layers to predict CL, CD, CM
        self.bn2 = nn.BatchNorm1d(256)
        self.drop2 = nn.Dropout(0.4)
        self.fc3 = nn.Linear(256, 3) # Fully Connected Dense layers to predict CL, CD, CM

    def forward(self, xyz, re_input):
        B, C, N = xyz.shape
        z = torch.zeros((B, 1, N), device=xyz.device)
        xyz_3d = torch.cat([xyz, z], dim=1)

        l1_xyz, l1_points = self.sa1(xyz_3d, None)
        l2_xyz, l2_points = self.sa2(l1_xyz, l1_points)
        l3_xyz, l3_points = self.sa3(l2_xyz, l2_points)

        global_feat = l3_points.view(B, 1024)
        physics_feat = torch.cat([global_feat, re_input], dim=1)

        x = F.relu(self.bn1(self.fc1(physics_feat)))
        x = self.drop1(x)
        x = F.relu(self.bn2(self.fc2(x)))
        x = self.drop2(x)
        scalars = self.fc3(x)

        physics_feat_expanded = physics_feat.unsqueeze(-1)
        l2_points = self.fp3(l2_xyz, l3_xyz, l2_points, physics_feat_expanded)
        l1_points = self.fp2(l1_xyz, l2_xyz, l1_points, l2_points)
        l0_points = self.fp1(xyz_3d, l1_xyz, xyz_3d, l1_points)

        cp_dist = self.conv_cp(l0_points).squeeze(1)
        return scalars, cp_dist


forward() is the forward pass of the entire AI. Batch of airfoil shapes (B×2×200), Batch of Reynolds numbers ($B \times 1$).

1. Encode: Passes data through sa1 $\to$ sa2 $\to$ sa3 to get global_feat (size 1024).
2. Physics Injection: torch.cat([global_feat, re_input]). Attaches the Reynolds number to the shape features. This tells the AI which flight condition it is simulating.
3. Scalar Prediction: Passes this combined vector through fc1, fc2, fc3 to guess Lift, Drag, and Moment.
4. Decode: Passes features back up through fp3, fp2, fp1.
5. Curve Prediction: Runs the final dense features through conv_cp to get the pressure distribution curve.

Returns: scalars (Lift/Drag/Moment) and cp_dist (Pressure Curve).

In [ ]:
class CST_Kernel:
    def __init__(self, n_params=8, resolution=200):
        self.n_params = n_params
        self.resolution = resolution
        self.beta = np.linspace(0, np.pi, self.resolution) # cosine spacing
        self.x = 0.5 * (1 - np.cos(self.beta)) # x coordinates clustered at LE and TE
        self.B = np.zeros((self.resolution, self.n_params)) # Bernstein Basis Matrix (maps 8 weights to 200 surface points.)
        self.C = np.sqrt(self.x) * (1 - self.x) # Class function which forces the shape to look like an airfoil (round nose, sharp tail).
        n = self.n_params - 1
        for k in range(self.n_params):
            self.B[:, k] = comb(n, k) * (self.x**k) * ((1 - self.x)**(n - k))

    def compute(self, w_u, w_l, dz): # Generates a specific airfoil from weights.
        S_u = self.B @ w_u
        S_l = self.B @ w_l
        y_u = self.C * S_u + self.x * (dz / 2.0)
        y_l = -self.C * S_l - self.x * (dz / 2.0)

        # 1. Thickness Check
        thickness = y_u - y_l
        if np.any(thickness < -1e-6): return None, 0, 0

        # 2. Leading Edge Radius Approximation (Simple)
        # Radius ~ proportional to sqrt(w_u[0])
        # We ensure w_u[0] (Class shape at LE) is large enough
        le_metric = w_u[0] + w_l[0]

        # 3. Curvature Energy (Smoothness)
        # Minimize 2nd derivative energy
        k_u = np.gradient(np.gradient(y_u))
        k_l = np.gradient(np.gradient(y_l))
        energy = np.sum(k_u**2) + np.sum(k_l**2)

        x_coords = np.concatenate((self.x[::-1], self.x[1:]))
        y_coords = np.concatenate((y_u[::-1], y_l[1:]))
        points = np.column_stack((x_coords, y_coords))

        if len(points) != 200:
             idx = np.linspace(0, len(points)-1, 200)
             px = np.interp(idx, np.arange(len(points)), points[:,0])
             py = np.interp(idx, np.arange(len(points)), points[:,1])
             points = np.column_stack((px, py))

        return points.astype(np.float32), le_metric, energy

class NeuralOptimizer:
    def __init__(self):
        self.kernel = CST_Kernel()

        print(f"--- Loading Model: {CONFIG['MODEL_PATH']} ---")
        self.model = ReAwareAirfoilPointNet().to(CONFIG['DEVICE'])
        try:
            self.model.load_state_dict(torch.load(CONFIG['MODEL_PATH'], map_location=CONFIG['DEVICE']))
        except Exception as e:
            print(f"Error loading model: {e}"); sys.exit(1)
        self.model.eval()

        print("--- Calibrating Scalers ---")
        if not os.path.exists(CONFIG['DATASET_PATH']):
            print("Dataset missing."); sys.exit(1)

        with h5py.File(CONFIG['DATASET_PATH'], 'r') as f:
            scalars = f['scalars'][:]
        valid_mask = np.isfinite(scalars).all(axis=1)
        scalars = scalars[valid_mask]

        self.target_scaler = StandardScaler()
        self.target_scaler.fit(scalars[:, 0:3])

        self.re_scaler = MinMaxScaler()
        self.re_scaler.fit(scalars[:, 4].reshape(-1, 1))

        re_val = np.array([[CONFIG['TARGET_REYNOLDS']]])
        self.re_norm = torch.tensor(self.re_scaler.transform(re_val), dtype=torch.float32).to(CONFIG['DEVICE'])

    def evaluate(self, population): #Scores a batch of 1000 airfoils.
        valid_pop, point_clouds, smoothness_penalties = [], [], []

        for ind in population:
            # Generate shape and GET GEOMETRIC METRICS
            pc, le_metric, energy = self.kernel.compute(ind['w_u'], ind['w_l'], ind['dz'])

            if pc is not None:
                # CONSTRAINT 1: Thickness
                t_max = np.max(pc[:, 1]) - np.min(pc[:, 1])

                # CONSTRAINT 2: Round Nose (Prevents Sharp Stall)
                # w[0] roughly correlates to nose radius in CST
                # We enforce a minimum LE width parameter
                if t_max >= CONFIG['MIN_THICKNESS'] and le_metric > CONFIG['MIN_LE_RADIUS']:

                    point_clouds.append(pc.T)
                    valid_pop.append(ind)

                    # Calculate penalty for waviness
                    # If energy > max, apply linear penalty
                    pen = max(0, energy - CONFIG['MAX_CURVATURE_ENERGY']) * 0.1
                    smoothness_penalties.append(pen)

        if not point_clouds: return []

        batch_pc = torch.tensor(np.array(point_clouds), dtype=torch.float32).to(CONFIG['DEVICE'])
        batch_re = self.re_norm.repeat(len(batch_pc), 1)

        with torch.no_grad():
            scalars_pred, _ = self.model(batch_pc, batch_re)

        real_vals = self.target_scaler.inverse_transform(scalars_pred.cpu().numpy())

        for i, vals in enumerate(real_vals):
            cl, cd, cm = vals

            # --- ROBUST FITNESS FUNCTION ---
            # 1. Minimize Drag (Primary)
            # 2. Minimize Pitching Moment (Secondary - Improves Stability)
            # 3. Penalize Lift Miss (Constraint)
            # 4. Penalize Wavy Surfaces (Geometric Constraint)

            score_drag = - (cd * 100.0)
            score_moment = - (abs(cm) * 10.0) # Penalty for instability
            score_lift = - (max(0, CONFIG['TARGET_CL'] - cl) * 50.0)
            score_smooth = - smoothness_penalties[i]

            fitness = score_drag + score_moment + score_lift + score_smooth

            valid_pop[i].update({'cl': float(cl), 'cd': float(cd), 'cm': float(cm), 'fitness': fitness})

        return valid_pop

    def run(self):
        # Init population with slightly larger LE weights for round noses
        pop = []
        for _ in range(CONFIG['POPULATION_SIZE']):
            w_u = np.random.uniform(-0.1, 0.4, 8)
            w_l = np.random.uniform(-0.2, 0.2, 8)
            w_u[0] = abs(w_u[0]) + 0.1 # Biased towards round nose
            w_l[0] = abs(w_l[0]) + 0.1
            pop.append({'w_u': w_u, 'w_l': w_l, 'dz': 0.005})

        print(f"\nROBUST OPTIMIZATION -> Re: {CONFIG['TARGET_REYNOLDS']:.0f} | Round Nose Enforced")

        best_overall = None
        for gen in range(CONFIG['GENERATIONS']):
            eval_pop = self.evaluate(pop)
            if not eval_pop: continue

            eval_pop.sort(key=lambda x: x['fitness'], reverse=True)
            if best_overall is None or eval_pop[0]['fitness'] > best_overall['fitness']:
                best_overall = eval_pop[0]

            if gen % 10 == 0:
                print(f"Gen {gen:03d} | CL:{best_overall['cl']:.2f} CD:{best_overall['cd']:.5f} CM:{best_overall['cm']:.3f}")

            next_gen = eval_pop[:50]
            while len(next_gen) < CONFIG['POPULATION_SIZE']:
                p1, p2 = random.choice(eval_pop[:100]), random.choice(eval_pop[:100])
                alpha = random.random()

                # Smoother Mutation
                child_wu = p1['w_u']*alpha + p2['w_u']*(1-alpha) + np.random.normal(0, 0.01, 8)
                child_wl = p1['w_l']*alpha + p2['w_l']*(1-alpha) + np.random.normal(0, 0.01, 8)

                # Keep Nose Round
                child_wu[0] = max(0.05, child_wu[0])
                child_wl[0] = max(0.05, child_wl[0])

                next_gen.append({'w_u': child_wu, 'w_l': child_wl, 'dz': (p1['dz'] + p2['dz'])/2.0})
            pop = next_gen

        return best_overall # After 150 generations.

In [ ]:
def perform_alpha_sweep(ind):
    xfoil = shutil.which("xfoil.exe") or "xfoil"
    if not xfoil: print("XFOIL not found."); return

    print("\n--- Running XFOIL Robustness Check (0-16 deg) ---")
    kernel = CST_Kernel()
    pts, _, _ = kernel.compute(ind['w_u'], ind['w_l'], ind['dz'])

    with open("robust_airfoil.dat", "w") as f:
        f.write("ROBUST_AF\n")
        for p in pts: f.write(f" {p[0]:.6f}  {p[1]:.6f}\n")

    polar_file = os.path.abspath("robust_polar.dat")
    if os.path.exists(polar_file): os.remove(polar_file)

    with tempfile.TemporaryDirectory() as tmp:
        shutil.copy("robust_airfoil.dat", os.path.join(tmp, "run.dat"))
        cmds = (
            f"load run.dat\n"
            "ppar\n N 200\n pane\n \n \n"
            "oper\n"
            f"v {CONFIG['TARGET_REYNOLDS']}\n"
            "iter 150\n" # More iterations for convergence
            "pacc\n"
            f"{polar_file}\n \n"
            "aseq 0 16 0.5\n" # Finer resolution
            "pacc\n"
            "quit\n"
        )
        subprocess.run(xfoil, input=cmds, text=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, cwd=tmp)

    if os.path.exists(polar_file):
        print(f"\n{'Alpha':<8} {'CL':<8} {'CD':<8} {'CM':<8}")
        print("-" * 45)
        cl_vals, cd_vals = [], []
        with open(polar_file, 'r') as f:
            for line in f:
                if any(c.isalpha() for c in line[:5]): continue
                vals = line.split()
                if len(vals) > 4:
                    try:
                        a, cl, cd, cm = float(vals[0]), float(vals[1]), float(vals[2]), float(vals[4])
                        print(f"{a:<8.1f} {cl:<8.4f} {cd:<8.5f} {cm:<8.4f}")
                        cl_vals.append(cl)
                        cd_vals.append(cd)
                    except: pass

        # Plot for User Visual Check
        plt.figure(figsize=(6,6))
        plt.plot(cd_vals, cl_vals, 'b-o')
        plt.xlabel('Cd')
        plt.ylabel('Cl')
        plt.title('Drag Polar')
        plt.grid(True)
        plt.savefig("robust_polar_plot.png")
        print("\n [Image of Polar Curve]Saved to 'robust_polar_plot.png'. Check for smoothness.")

In [ ]:
if __name__ == "__main__":
    opt = NeuralOptimizer()
    best = opt.run()
    if best: perform_alpha_sweep(best)